In [ ]:
import scanpy as sc, anndata as ad, numpy as np, pandas as pd
import anndata2ri
import rpy2.rinterface_lib.callbacks
import logging
from matplotlib import pylab
import os
import sys
anndata2ri.activate()
import yaml
%reload_ext rpy2.ipython

from scipy.sparse import csr_matrix, isspmatrix


In [ ]:
pylab.rcParams['figure.figsize'] = (9, 9)
homeDir = os.getenv("HOME")
sys.path.insert(1, homeDir+"/utils/")
from PlotPCA_components import *
from AdataSanityCheck import *
from PurgeAdata import *
from spatialUtils import *
from _DEAplots import *
from _Aggregation import *
from _plotting import *

In [ ]:
with open(homeDir+"/utils/config.yaml", 'r') as f:
    analysis_params = yaml.safe_load(f)["analysisParams"]
print(analysis_params)

# Load data

In [ ]:
adata = sc.read_h5ad(os.path.join(homeDir, "1_DataPreparation/out/0_Filtered.h5ad"))
adata = adata.copy()

# We downsample condition to have roughly same cell numbers

In [ ]:
import random

seed = 0
rng = random.Random(seed)

minCells = adata.obs["condition"].value_counts().min()

BCs = []
for cond in sorted(adata.obs["condition"].astype(str).unique()):
    idx = adata.obs_names[adata.obs["condition"].astype(str) == cond].tolist()
    BCs.extend(rng.sample(idx, minCells))

adata[BCs].obs["condition"].value_counts()


In [ ]:
adata = adata[BCs].copy()

In [ ]:
import pandas as pd

# Convert all string columns in .obs to native strings
for col in adata.obs.columns:
    if pd.api.types.is_string_dtype(adata.obs[col]):
        adata.obs[col] = adata.obs[col].astype(str)

# Convert all string columns in .var to native strings
for col in adata.var.columns:
    if pd.api.types.is_string_dtype(adata.var[col]):
        adata.var[col] = adata.var[col].astype(str)

# Also convert the index, which is where your error is actually coming from
if pd.api.types.is_string_dtype(adata.var.index):
    adata.var.index = adata.var.index.astype(str)

if pd.api.types.is_string_dtype(adata.obs.index):
    adata.obs.index = adata.obs.index.astype(str)




In [ ]:
adata.layers["counts"] = adata.X.copy()


In [ ]:
sc.pp.normalize_total(adata)
sc.pp.log1p(adata)



Mt0_1 = adata[adata.obs["condition"].isin(["Melanoma","Melanoma_96Hrs_CoColture"])].copy()
sc.pp.highly_variable_genes(Mt0_1, flavor="seurat", subset=False,n_top_genes=1500)
Mt0_HVGs = set(Mt0_1.var_names[Mt0_1.var["highly_variable"]])
print(len(Mt0_HVGs))


Mt1_2 = adata[adata.obs["condition"].isin(["Melanoma_96Hrs_CoColture","Melanoma_2Weeks_CoColture"])].copy()
sc.pp.highly_variable_genes(Mt1_2, flavor="seurat", subset=False,n_top_genes=1500)
Mt1_HVGs = set(Mt1_2.var_names[Mt1_2.var["highly_variable"]])
print(len(Mt1_HVGs))

Mt2_3 = adata[adata.obs["condition"].isin(["Melanoma_2Weeks_CoColture","Melanoma_1Month_CoColture"])].copy()
sc.pp.highly_variable_genes(Mt2_3, flavor="seurat", subset=False,n_top_genes=1500)
Mt2_HVGs = set(Mt2_3.var_names[Mt2_3.var["highly_variable"]])
print(len(Mt2_HVGs))


HVGs = list(Mt0_HVGs.union(Mt1_HVGs).union(Mt2_HVGs))
adata.var["HVG_sequential"] = adata.var_names.isin(HVGs)
print(adata.var["HVG_sequential"].sum())



sc.pp.pca(adata, mask_var="HVG_sequential")
sc.pl.pca(adata, color=["condition"], frameon=False, wspace=0.4, hspace=0.4, ncols=2)

plotPCA_components(adata, color="condition")

sc.pl.pca_variance_ratio(adata)

In [ ]:
sc.pp.neighbors(adata, n_neighbors=50, n_pcs=15)
sc.tl.umap(adata)

In [ ]:
sc.tl.leiden(adata, flavor="igraph", resolution=.6)
if "leiden_colors" in adata.uns:
    del adata.uns["leiden_colors"]

In [ ]:
sc.pl.umap(adata, color=["condition", "TOP2A","STMN2","leiden"], size=20, ncols=2, wspace=.4)

In [ ]:
plotResiduals(adata, npcs=5, max_points=10000, covToTest=["condition","leiden"])

In [ ]:
sc.tl.rank_genes_groups(adata, groupby="leiden")

In [ ]:
sc.pl.rank_genes_groups_dotplot(
    adata,
    n_genes=10,
    values_to_plot="logfoldchanges",
    min_logfoldchange=1,
    vmax=3,
    vmin=-3,
    cmap="bwr",
)

In [ ]:
sc.pl.rank_genes_groups(
    adata,
    n_genes=30, fontsize=10
)

# Metacell-based method

# First check number of genes expressed per gorup

In [ ]:
import numpy as np
import pandas as pd
import scipy.sparse as sp

start, stop, step = 0, 1, 0.05
Rates = np.round(np.arange(start, stop + 1e-12, step), 2)

rows = []
for cl in adata.obs["leiden"].unique():
    local = adata[adata.obs["leiden"] == cl]
    X = local.X
    n_cells = local.n_obs

    # number of positive cells per gene
    if sp.issparse(X):
        pos_counts = np.asarray((X > 0).sum(axis=0)).ravel()
    else:
        pos_counts = (X > 0).sum(axis=0).ravel()

    # inclusive thresholds: need >= ceil(n_cells * rate)
    thr = np.ceil(n_cells * Rates).astype(int)
    thr[Rates == 0.0] = 1  # define 0.00 as ">=1 cell positive"

    PosList = (pos_counts[None, :] >= thr[:, None]).sum(axis=1)
    rows.append(pd.Series(PosList, index=Rates, name=cl))

PosCellsDF = pd.DataFrame(rows)   # rows=clusters, cols=Rates
PosCellsDF

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import math


import numpy as np
import pandas as pd
import scipy.sparse as sp

start, stop, step = 0, 1, 0.05
Rates = np.round(np.arange(start, stop + 1e-12, step), 2)

rows = []
for cl in adata.obs["leiden"].unique():
    local = adata[adata.obs["leiden"] == cl]
    X = local.X
    n_cells = local.n_obs

    # number of positive cells per gene
    if sp.issparse(X):
        pos_counts = np.asarray((X > 0).sum(axis=0)).ravel()
    else:
        pos_counts = (X > 0).sum(axis=0).ravel()

    # inclusive thresholds: need >= ceil(n_cells * rate)
    thr = np.ceil(n_cells * Rates).astype(int)
    thr[Rates == 0.0] = 1  # define 0.00 as ">=1 cell positive"

    PosList = (pos_counts[None, :] >= thr[:, None]).sum(axis=1)
    rows.append(pd.Series(PosList, index=Rates, name=cl))

PosCellsDF = pd.DataFrame(rows)   # rows=clusters, cols=Rates



# Ensure posrate is numeric + sorted
Rates = np.array(PosCellsDF.columns, dtype=float)
PosCellsDF = PosCellsDF.loc[:, Rates[np.argsort(Rates)]]
Rates = np.array(PosCellsDF.columns, dtype=float)

clusters = list(PosCellsDF.index)
n = len(clusters)

# number of cells per cluster (needs adata in scope)
cell_counts = (
    adata.obs["leiden"]
    .value_counts()
    .reindex(clusters)
    .astype(int)
)

# layout
ncols = min(2, n)
nrows = math.ceil(n / ncols)

step = float(np.median(np.diff(Rates))) if len(Rates) > 1 else 0.05
bar_w = 0.9 * step

ymax = float(np.nanmax(PosCellsDF.to_numpy()))

fig, axes = plt.subplots(
    nrows, ncols,
    figsize=(6 * ncols, 2.8 * nrows),
    sharex=True, sharey=True
)
axes = np.atleast_1d(axes).ravel()

for i, cl in enumerate(clusters):
    ax = axes[i]
    y = PosCellsDF.loc[cl].to_numpy(dtype=float)

    bars = ax.bar(Rates, y, width=bar_w, align="center")
    ax.set_title(f"{cl} (n={cell_counts.loc[cl]})")
    ax.set_ylim(0, ymax * 1.05)

    # annotate #positive genes (bar height) on top of each bar
    for rect, val in zip(bars, y):
        ax.text(
            rect.get_x() + rect.get_width() / 2,
            rect.get_height() + ymax * 0.01,
            f"{int(val)}",
            ha="center",
            va="bottom",
            rotation=90,
            fontsize=6,
            clip_on=True
        )

# hide unused axes
for j in range(n, len(axes)):
    axes[j].axis("off")

# ticks/labels
for ax in axes[:n]:
    ax.set_xticks(Rates)
    ax.set_xticklabels([f"{r:.2f}" for r in Rates], rotation=90, ha="center")

# axis labels
for r in range(nrows):
    axes[r * ncols].set_ylabel("# genes")

for ax in axes[(nrows - 1) * ncols : nrows * ncols]:
    if ax.has_data():
        ax.set_xlabel("posrate")

fig.tight_layout()
plt.show()


In [ ]:
import numpy as np
from scipy import sparse

MinPositivePerClusterRate = 0.5
clusters = adata.obs["leiden"]

# If cluster is categorical, this preserves a stable order
cluster_levels = clusters.cat.categories if hasattr(clusters.dtype, "categories") else clusters.unique()

n_vars = adata.n_vars
keep_mask = np.zeros(n_vars, dtype=bool)

for cl in cluster_levels:
    idx = (clusters == cl).to_numpy()
    n = int(idx.sum())
    if n == 0:
        continue

    Xc = adata.X[idx, :]  # slice once

    if sparse.issparse(Xc):
        pos_counts = np.asarray((Xc > 0).sum(axis=0)).ravel()
    else:
        pos_counts = (Xc > 0).sum(axis=0)

    keep_mask |= (pos_counts >= MinPositivePerClusterRate * n)

KeptGenes = adata.var_names[keep_mask].tolist()
print(len(KeptGenes))

In [ ]:
metacellsAdata = adata.copy()
metacellsAdata.X = metacellsAdata.layers["counts"]
sc.pp.normalize_total(metacellsAdata)

metacellsAdata
metacellsAdata = deterministic_aggregation_k3(
    _adata_group = metacellsAdata,
    group="Myeloids",
    cellStateObs = "leiden",
    PseudoReplicates_per_group = 5,pca="X_pca",
    method="k3Metacells",countsLayer=None,
    n_pcs=15)


In [ ]:
metacellsAdata.layers

In [ ]:
def sanitize_strings(value):
    if isinstance(value, str):
        return (
            value.replace('_', '')
            .replace('-', '')
            .replace('@', '')
            .replace(',', '')
            .replace('/', '')
            .replace(':', '')
            .replace('(', '')
            .replace(')', '')
            .replace(';', '')
            .replace(' ', '')
        )
    return value


metacellsAdata.X = metacellsAdata.layers["k3Metacells_meanCounts"].copy()
#sc.pp.normalize_total(metacellsAdata, 1e6)
Counts = metacellsAdata.to_df().T.copy()
Counts = Counts.loc[KeptGenes]


MD = metacellsAdata.obs.copy()
MD["cluster"] = MD["leiden"].astype(str)

# 1) Apply to all values in MD (every column)
MD = MD.applymap(sanitize_strings)

# 2) Sanitize column names and index of MD
MD.columns = MD.columns.map(sanitize_strings)
MD.index   = MD.index.map(sanitize_strings)

# 3) Sanitize column names and index of Counts
Counts.columns = Counts.columns.map(sanitize_strings)



MD["cluster"].value_counts()


In [ ]:
%%R -i MD -i Counts -o results -o genes



library(edgeR)
library(org.Hs.eg.db)
library(AnnotationDbi)
library(stats)


cluster <- factor(MD[["cluster"]])

y <- DGEList(Counts, group = cluster, samples=rownames(MD), genes = rownames(Counts))






cluster <- as.factor(y$samples$group)
donor <- factor(y$samples$sample)
design <- model.matrix(~ 0+cluster )
colnames(design) <- gsub("cluster", "", colnames(design))

print(head(design))





print("estimateDisp")
y <- estimateDisp(y, design, robust=TRUE)
print("fitting")

fit <- glmQLFit(y, design, robust=TRUE)




ncls <- nlevels(cluster)
contr <- rbind( matrix(1/(1-ncls), ncls, ncls),matrix(0, ncol(design)-ncls, ncls) )
diag(contr) <- 1
rownames(contr) <- colnames(design)
colnames(contr) <- levels(cluster)
contr


AllGenes <- 30000
results <- list()
for(i in colnames(contr)){
    print(sprintf("Extracting toptags for  %s",i ))
    qlf <-glmQLFTest(fit, contrast=contr[,i])
    qlf <- topTags(qlf, n=AllGenes, sort.by="logFC")$table
    results[[i]] <- qlf
}


genes <- rownames(y)

In [ ]:
resultsDict = dict(zip([str(i) for i in  list(results.names())], list(results.values())))
for k in resultsDict:
    resultsDict[k]["celltype"] = k
    resultsDict[k]["method"] = "DEA"
    resultsDict[k]["genes"] = resultsDict[k].index.tolist()
    resultsDict[k]["rank"] = np.sign(resultsDict[k]["logFC"])* -np.log10(resultsDict[k]["FDR"])
    print(k)

agg_df, DEGsDictGO, fig = faceted_volcano(
    results_dict=resultsDict,
    fc="logFC", p="PValue", q="FDR", gene_col="genes",
    logfc_threshold=0.5, q_threshold=0.01,marker_size=8,width_scaleF=400,marker_line_width=0.4,
    q_trim=1, logfc_trim=np.inf,
    facet_col_wrap=4,  # change to 3/2 to make facets larger
    show=True
)


In [ ]:
import numpy as np
import pandas as pd
from scipy import sparse

topN = 10
markersDict = {}
stats_list = []
seenMarkers = set()

# Cache gene availability once (prevents AnnData slicing errors)
allowed_genes = set(adata.var_names)

def pick_top_unique_positive(
    df,
    topN,
    seen,
    allowed_genes,
    fdr_col="FDR",
    rank_col="rank",
    logfc_col="logFC",
    minlogFC = 1,
    fdr_thresh=0.01
):
    """
    Picks topN unique UP (positive) markers:
      - requires FDR < fdr_thresh
      - requires logFC > 0 if available, otherwise rank > 0
      - sorts by rank descending (bigger positive = more significant up)
    """
    # Filter by FDR first
    df2 = df.loc[df[fdr_col] < fdr_thresh].copy()

    # Enforce "positive" direction
    if logfc_col in df2.columns:
        df2 = df2.loc[df2[logfc_col] > minlogFC]
    else:
        df2 = df2.loc[df2[rank_col] > 0]

    # Sort: biggest positive rank first
    df2 = df2.sort_values(logfc_col, ascending=False)

    picked = []
    for g in df2.index:
        if g not in allowed_genes:
            continue
        if g in seen:
            continue
        picked.append(g)
        seen.add(g)
        if len(picked) == topN:
            break
    return picked

# categories for contrast groups
contrast = adata.obs["leiden"].astype("category")
groups = list(contrast.cat.categories)

# Precompute indices per group once (much faster)
contrast_vals = contrast.to_numpy()
group_indices = {g: np.where(contrast_vals == g)[0] for g in groups}

for clName, df in resultsDict.items():
    # (Optional but recommended) robustify rank against FDR==0 if needed
    # eps = 1e-300
    # df = df.copy()
    # df["rank"] = np.sign(df["logFC"]) * -np.log10(np.maximum(df["FDR"], eps))

    markers = pick_top_unique_positive(df, topN, seenMarkers, allowed_genes)
    markersDict[clName] = markers
    if not markers:
        continue

    # cells x genes matrix (keeps marker order)
    X = adata[:, markers].X
    if sparse.issparse(X):
        X = X.tocsr()

    for g, idx in group_indices.items():
        if idx.size == 0:
            continue

        Xg = X[idx, :]
        n_cells = idx.size

        if sparse.issparse(Xg):
            # mean per gene
            mean_expr = np.asarray(Xg.mean(axis=0)).ravel()
            # fraction > 0 per gene (fast for sparse)
            pos_rate = np.asarray(Xg.getnnz(axis=0)).ravel() / n_cells
        else:
            Xg_arr = np.asarray(Xg)
            mean_expr = Xg_arr.mean(axis=0)
            pos_rate = (Xg_arr > 0).mean(axis=0)

        stats_list.append(pd.DataFrame({
            "leiden": [g] * len(markers),
            "gene": markers,
            "positive_rate": pos_rate,
            "mean_expression": mean_expr,
            "n_cells": [n_cells] * len(markers),
            "cluster": [clName] * len(markers),
        }))

stats_df = pd.concat(stats_list, ignore_index=True) if stats_list else pd.DataFrame()


In [ ]:
import itertools
DEGs = list(itertools.chain(*list(markersDict.values())))
adata.X = adata.layers["counts"].copy()
sc.pp.normalize_total(adata)
sc.pp.log1p(adata)
adataMyeloids = adata[~adata.obs["leiden"].str.contains("_Niche"),DEGs].copy()
ax = sc.pl.dotplot(adataMyeloids, markersDict, groupby="leiden", swap_axes=False, cmap="bwr",#categories_order=BanksyDomainOrder,
                          dendrogram=False,standard_scale ="var")

In [ ]:
def sanitize_adata(adata):
    import pandas as pd
    import numpy as np
    from scipy.sparse import issparse, csr_matrix

    def convert_df(df, name=""):
        for col in df.columns:
            dtype = df[col].dtype

            if pd.api.types.is_integer_dtype(dtype):
                print(f"[{name}] Converting column '{col}' from {dtype} → float32")
                df[col] = df[col].astype(np.float32)

            elif pd.api.types.is_categorical_dtype(dtype):
                print(f"[{name}] Converting column '{col}' from category → string")
                df[col] = df[col].astype(str)

        df.columns = df.columns.astype(str)
        return df

    # Sanitize obs and var
    adata.obs = convert_df(adata.obs, name="obs")
    adata.var = convert_df(adata.var, name="var")
    adata.obs_names = adata.obs_names.astype(str)
    adata.var_names = adata.var_names.astype(str)

    # Sanitize uns colors
    for unskey in [k for k in adata.uns if "_colors" in k]:
        if not isinstance(adata.uns[unskey], list):
            adata.uns[unskey] = adata.uns[unskey].astype(str).tolist()

    # Sanitize .X
    if not issparse(adata.X):
        print("Converting adata.X to csr_matrix")
        adata.X = csr_matrix(adata.X)
    if adata.X.dtype != np.float32:
        print(f"Converting adata.X dtype from {adata.X.dtype} to float32")
        adata.X = adata.X.astype(np.float32)

    # Sanitize layers
    for layer in list(adata.layers.keys()):
        if not issparse(adata.layers[layer]):
            print(f"Converting adata.layers[{layer}] to csr_matrix")
            adata.layers[layer] = csr_matrix(adata.layers[layer])
        if adata.layers[layer].dtype != np.float32:
            print(f"Converting adata.layers[{layer}] dtype from {adata.layers[layer].dtype} to float32")
            adata.layers[layer] = adata.layers[layer].astype(np.float32)


In [ ]:
check_anndata_sanity(adata)
sanitize_adata(adata)

In [ ]:
adata.write_h5ad("./MelanomaCoCulture_processed.h5ad")